# MedCLIP-SAMv2 + MuscleMap Boxes — Lambda

Runs MedSAM on both **water** and **fat-fraction** Dixon stacks, using
**MuscleMap WB bounding boxes** as prompts (one box per muscle per slice).

This is the same prompting strategy as `muscle_map_wb_boxes_medsam` but
is run here as a direct comparison within the MedCLIP-SAMv2 evaluation context.

MedSAM embedding is computed **once per slice** and reused for all muscles.

The MuscleMap WB segmentations are expected to already be on Lambda from
previous runs:
- Water: `~/musclemap_water_segs/` (from `lambda_musclemap_plus_medsam_water.ipynb`)
- Fat fraction: `~/MuscleMap_segs/` (from `lambda_musclemap_plus_medsam_logitmask.ipynb`)

If they are not present, upload them first:

```bash
# MuscleMap WB water segmentations
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/segs_water/ \
  your machine9:~/musclemap_water_segs/

# MuscleMap WB fat-fraction segmentations
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/segmentations_fat_frac/ \
  your machine9:~/MuscleMap_segs/

# MedSAM checkpoint (if not already present)
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  your machine9:~/medsam_vit_b.pth
```

## Download results when done

```bash
# water
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine9:~/medclipsamv2_plusboxes_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2plusboxes/segs_water/

# fat fraction
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine9:~/medclipsamv2_plusboxes_ff/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medclipsamv2plusboxes/segmentations_fat_frac/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'scikit-image'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob
import os
import re
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from skimage import transform
from segment_anything import sam_model_registry

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SAM_DEVICE = 'cpu'   # MedSAM encoder is memory-heavy; keep on CPU

# ── paths ─────────────────────────────────────────────────────────────────────
SAM_CKPT = os.path.expanduser('~/medsam_vit_b.pth')

# MuscleMap WB segmentation directories — same locations used by the
# existing musclemap lambda notebooks
MM_WATER_DIR = os.path.expanduser('~/musclemap_water_segs')   # water _dseg.nii.gz
MM_FF_DIR    = os.path.expanduser('~/MuscleMap_segs')          # fat-fraction _dseg.nii.gz

# Raw image globs
WATER_GLOB = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_WATER/*_WATER_stack*.nii')
FF_GLOB    = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_FATFRACTION/*_FATFRACTION_stack*.nii')

# Output directories
OUT_WATER = os.path.expanduser('~/medclipsamv2_plusboxes_water')
OUT_FF    = os.path.expanduser('~/medclipsamv2_plusboxes_ff')
os.makedirs(OUT_WATER, exist_ok=True)
os.makedirs(OUT_FF,    exist_ok=True)

if not os.path.exists(SAM_CKPT):
    raise FileNotFoundError(f'MedSAM checkpoint not found: {SAM_CKPT} — upload it first.')

print('SAM device  :', SAM_DEVICE)
print('MM water dir:', MM_WATER_DIR, '— exists:', os.path.isdir(MM_WATER_DIR))
print('MM FF dir   :', MM_FF_DIR,    '— exists:', os.path.isdir(MM_FF_DIR))
print('Out water   :', OUT_WATER)
print('Out FF      :', OUT_FF)

In [ ]:
# MuscleMap WB label map — all 20 thigh muscles
MM_WB_LABELS = {
    7101: 'Vastus_Lateralis_L',   7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L', 7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',    7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',     7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',          7142: 'Sartorius_R',
    7151: 'Gracilis_L',           7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',    7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',     7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',     7182: 'Biceps_Femoris_R',
    7201: 'Adductor_Magnus_L',    7202: 'Adductor_Magnus_R',
}
print(f'{len(MM_WB_LABELS)} MuscleMap WB labels defined')

In [ ]:
def enlarge_bounding_box(mask, margin=5):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = mask.shape
    return np.array([
        max(0, cmin - margin), max(0, rmin - margin),
        min(W - 1, cmax + margin), min(H - 1, rmax + margin),
    ], dtype=float)


def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if box_torch.ndim == 2:
        box_torch = box_torch[:, None, :]
    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None, boxes=box_torch, masks=None
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )
    low_res_pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W),
        mode='bilinear', align_corners=False
    )
    return (low_res_pred.squeeze().detach().cpu().numpy() > 0.5).astype(np.uint8)


def process_stack(nii_path, seg_path, out_path, sam_model):
    """
    Run MedSAM on one image stack using MuscleMap WB boxes as prompts.
    Saves a compressed NPZ with one array per muscle name.
    """
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    H, W      = img_array.shape[1], img_array.shape[2]
    print(f'  Image : {img_array.shape}')

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)
    print(f'  Seg   : {seg_array.shape}')

    all_masks = {}

    for slice_idx in range(img_array.shape[0]):
        slice_2d  = img_array[slice_idx]
        seg_slice = seg_array[slice_idx]

        # MedSAM image embedding — once per slice, reused for all muscles
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)
        del img_tensor

        for label_idx, muscle_name in MM_WB_LABELS.items():
            mask_arr = (seg_slice == label_idx).astype(np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined

        del image_embedding

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]}')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved -> {out_path}')


print('Helpers defined.')

In [ ]:
# Load MedSAM once — reused for both modalities
sam_model = sam_model_registry['vit_b'](checkpoint=SAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
# ── Water ─────────────────────────────────────────────────────────────────────
water_files = sorted(glob.glob(WATER_GLOB))
print(f'Found {len(water_files)} water stacks')

matched_w, missing_w = [], []
for nii_path in water_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_WATER_DIR, f'{stem}_dseg.nii.gz')
    if os.path.exists(seg_path):
        matched_w.append((nii_path, seg_path))
    else:
        missing_w.append(stem)

print(f'  {len(matched_w)} matched, {len(missing_w)} missing MuscleMap seg')
if missing_w:
    print('  Missing:', missing_w[:5])

In [ ]:
for nii_path, seg_path in matched_w:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUT_WATER, f'{stem}_mcsam2boxes.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing (water): {stem}')
    process_stack(nii_path, seg_path, out_path, sam_model)

print('\nWater done.')

In [ ]:
# ── Fat Fraction ──────────────────────────────────────────────────────────────
ff_files = sorted(glob.glob(FF_GLOB))
print(f'Found {len(ff_files)} fat-fraction stacks')

matched_ff, missing_ff = [], []
for nii_path in ff_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_FF_DIR, f'{stem}_dseg.nii.gz')
    if os.path.exists(seg_path):
        matched_ff.append((nii_path, seg_path))
    else:
        missing_ff.append(stem)

print(f'  {len(matched_ff)} matched, {len(missing_ff)} missing MuscleMap seg')
if missing_ff:
    print('  Missing:', missing_ff[:5])

In [ ]:
for nii_path, seg_path in matched_ff:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUT_FF, f'{stem}_mcsam2boxes.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing (fat fraction): {stem}')
    process_stack(nii_path, seg_path, out_path, sam_model)

print('\nFat fraction done.')

In [ ]:
# Sanity check
for label, out_dir in [('water', OUT_WATER), ('fat fraction', OUT_FF)]:
    results = sorted(glob.glob(os.path.join(out_dir, '*.npz')))
    print(f'\n{label}: {len(results)} NPZ files')
    if results:
        sample = np.load(results[0])
        print(f'  Sample: {os.path.basename(results[0])}')
        for k in list(sample.files)[:4]:
            print(f'    {k}: shape={sample[k].shape}  voxels={int(sample[k].sum()):,}')